In [3]:
import pandas as pd
from pathlib import Path
from typing import List


# ==========================================
# CONFIG
# ==========================================

DATA_DIR = Path("../data/processed/openmeteo")

CITIES = [
    "Zurich", "Geneva", "Bern", "Basel", "Lausanne",
    "Lucerne", "St_Gallen", "Lugano", "Interlaken", "Central_CH"
]


# ==========================================
# PATH HELPERS
# ==========================================

def get_year_path(year: int) -> Path:
    return DATA_DIR / f"year={year}.parquet"


def list_existing_files() -> List[Path]:
    return sorted(DATA_DIR.glob("year=*.parquet"))


# ==========================================
# CORE PARSER
# ==========================================

def parse_openmeteo_responses(responses, hourly_vars):
    dfs = []

    for i, response in enumerate(responses):
        hourly = response.Hourly()

        df = pd.DataFrame({
            "timestamp_utc": pd.date_range(
                start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
                end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
                freq=pd.Timedelta(seconds=hourly.Interval()),
                inclusive="left",
            ),
            "city": CITIES[i],
            "location_id": i,
            "latitude": response.Latitude(),
            "longitude": response.Longitude(),
            "elevation": response.Elevation(),
            "retrieved_at_utc": pd.Timestamp.utcnow(),
        })

        for j, var in enumerate(hourly_vars):
            df[var] = hourly.Variables(j).ValuesAsNumpy()

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


# ==========================================
# STORAGE LOGIC (KEY PART)
# ==========================================

def save_partitioned(df: pd.DataFrame):
    df["year"] = df["timestamp_utc"].dt.year

    for year, df_year in df.groupby("year"):
        path = get_year_path(year)

        if path.exists():
            old = pd.read_parquet(path)

            combined = (
                pd.concat([old, df_year])
                .drop_duplicates(subset=["timestamp_utc", "location_id"], keep="last")
                .sort_values("timestamp_utc")
            )
        else:
            combined = df_year.sort_values("timestamp_utc")

        path.parent.mkdir(parents=True, exist_ok=True)
        combined.to_parquet(path, index=False)

        print(f"Saved year={year} -> {path} | shape={combined.shape}")


# ==========================================
# LOAD FULL DATASET
# ==========================================

def load_full_dataset():
    files = list_existing_files()

    if not files:
        return pd.DataFrame()

    return pd.concat(
        [pd.read_parquet(f) for f in files],
        ignore_index=True
    )


# ==========================================
# MAIN PIPELINE
# ==========================================

def ingest_openmeteo(responses, hourly_vars):
    df = parse_openmeteo_responses(responses, hourly_vars)

    if df.empty:
        print("No data returned")
        return df

    save_partitioned(df)

    return load_full_dataset()

In [4]:
df = ingest_openmeteo(responses, HOURLY_VARS)

print(df.head())
print(df.shape)
print(df.dtypes)

NameError: name 'responses' is not defined

OpenMeteoRequestsError: failed to request 'https://historical-forecast-api.open-meteo.com/v1/forecast': {'reason': 'Hourly API request limit exceeded. Please try again in the next hour.', 'error': True}